In [1]:
# ============================================================
# INSTALL DEPENDENCIES
# Run this cell once before anything else.
# %pip ensures packages are installed into the active kernel.
# ============================================================

%pip install --quiet opencv-python numpy matplotlib tensorflow scikit-learn


/home/joshnelson/Documents/gcu/compsci/CST-440-MISC/CST-440/Project3/training/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ============================================================
# IMPORTS — All required libraries (Python 3.10+)
# Run this cell first every time you open the notebook.
# Packages must already be installed (pip install -r requirements.txt
# or run an install cell before this one if needed).
# ============================================================

# Standard library
import os
import sys
import subprocess
from pathlib import Path

# Numerical / image processing
import cv2
import numpy as np

# Visualisation
import matplotlib.pyplot as plt

# Machine learning framework
import tensorflow as tf
from tensorflow.keras import layers, models

# Dataset utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    precision_recall_fscore_support,
)

# ── Version check ─────────────────────────────────────────────
print(f"Python      : {sys.version}")
print(f"TensorFlow  : {tf.__version__}")
print(f"OpenCV      : {cv2.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"Working dir : {os.getcwd()}")


2026-03-03 16:02:44.652767: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-03 16:02:44.652996: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-03 16:02:44.684688: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-03 16:02:45.580022: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

Python      : 3.10.19 (main, Dec 17 2025, 21:07:02) [Clang 21.1.4 ]
TensorFlow  : 2.20.0
OpenCV      : 4.13.0
NumPy       : 2.2.6
Working dir : /home/joshnelson/Documents/gcu/compsci/CST-440-MISC/CST-440/Project3


# CST-440: Machine Learning on Microcontrollers
# Project 3: Face Detection on a Microcontroller

---

**Group:** Martin Battu, Josh Nelson, Emma Rogoveanu, Andru Tjalas  
**Course:** CST-440 Machine Learning on Microcontrollers  
**Platform:** Arduino Nano 33 BLE Sense + ArduCAM OV2640  
**Framework:** TensorFlow Lite for Microcontrollers  

---

## Table of Contents

1. [Introduction & Objective](#1.-Introduction-&-Objective)
2. [Data Preparation](#2.-Data-Preparation)
3. [Model Design](#3.-Model-Design)
4. [Training the Model](#4.-Training-the-Model)
5. [Evaluation](#5.-Evaluation)
6. [Model Conversion to TensorFlow Lite](#6.-Model-Conversion-to-TensorFlow-Lite)
7. [Testing the TensorFlow Lite Model](#7.-Testing-the-TensorFlow-Lite-Model)
8. [Conclusion](#8.-Conclusion)

---

> ⚠️ **Addendum — Dataset Not Included**
>
> The training dataset required to run this notebook is **not included** in this repository due to file-size constraints.
>
> Before executing any code cells beyond the imports, you must supply a dataset with the following structure:
>
> ```
> Project3/training/dataset/
> ├── face/        ← 96×96 grayscale face images (.jpg or .png)
> └── noface/      ← 96×96 grayscale non-face images (.jpg or .png)
> ```
>
> **To collect your own dataset**, use the data-collection helper in `Project3/training/` or source images manually. Images must be resized to **96×96 pixels** in **grayscale** to match the model's expected input (`IMG_SIZE = 96`).
>
> If the dataset directories are missing or empty, Cells 9, 10, 12, 15, 16, 18, 21, and 22 will fail. The pre-trained model files (`face_model.tflite`, `face_model_float32.tflite`) and the camera accuracy test (Cell 25) will still work independently of the dataset.

---


---

## 1. Introduction & Objective

### Project Overview

This project implements a **real-time binary face detection system** running entirely on an Arduino Nano 33 BLE Sense microcontroller. An ArduCAM OV2640 captures 160×120 RGB565 frames, which are cropped to 96×96 grayscale and classified as **FACE** or **NO FACE** using a quantized CNN — entirely on-device, with no cloud connection and no external compute at inference time.

### Objectives

1. Use the existing dataset in `Project3/face_model_dataset` (with `Project3/dataset` fallback)
2. Train a compact CNN matching the architecture in `train.py`
3. Quantize the model to INT8 TensorFlow Lite for deployment
4. Validate on a held-out test set and confirm ≥80% accuracy

### End-to-End Pipeline

```
OFFLINE (PC)
  Project3/face_model_dataset/face/    ← 96×96 face images   (label 1)
  Project3/face_model_dataset/noface/  ← 96×96 no-face images (label 0)
        ↓  train.py-equivalent notebook cells
  CNN training (binary cross-entropy, Adam, 20 epochs)
        ↓
  Project3/training/face_model.tflite          ← INT8, deployed to Arduino
  Project3/training/face_model_float32.tflite  ← float32, used by testing.py on PC
  Project3/training/face_model.h               ← C header for embedded code

ONLINE (Arduino Nano 33 BLE Sense)
  OV2640 160×120 RGB565 FIFO
  → center-crop 96×96 (CROP_X=32, CROP_Y=12)
  → RGB565→grayscale: Y = (77·R + 150·G + 29·B) >> 8  (BT.601 integer luma)
  → written directly into inputTensor->data.uint8  [no frame buffer needed]
  → TFLite Micro Invoke()
  → prob = (raw_uint8 - zero_point) × scale
  → prob ≥ 0.5 → Serial: "FACE DETECTED  prob=X.XXX  (N ms)"
  → prob < 0.5 → Serial: "No face detected  (N ms)"
```

### Hardware

| Component | Role |
|---|---|
| Arduino Nano 33 BLE Sense | ARM Cortex-M4 @ 64 MHz, 1 MB Flash, 256 KB SRAM |
| ArduCAM OV2640 Mini 2MP Plus | 160×120 RGB565 BMP mode via SPI FIFO + I2C config |
| PC | Dataset loading, training (`train.py`), desktop validation (`testing.py`) |

### Key Implementation Details from `main.cpp`

- **No frame buffer:** OV2640 FIFO is streamed pixel-by-pixel. Only the 96×96 center crop is written into the TFLite input tensor — saving ~38 KB of SRAM compared to buffering the full 160×120 frame.
- **Tensor arena:** 150 KB (`TENSOR_ARENA_KB=150` in `platformio.ini`)
- **Output dequantization:** `prob = (raw_uint8 - zero_point) × scale`, threshold at 0.5f
- **Both outcomes reported:** Every inference prints either `FACE DETECTED` (with probability and latency) or `No face detected` (with latency) to the Serial Monitor.


---

## 2. Data Preparation

### Dataset Structure

The notebook now loads an existing on-disk dataset (no webcam collection in notebook cells):

```
face_model_dataset/
├── face/     # 96×96 grayscale face images  — label 1
└── noface/   # 96×96 grayscale no-face images — label 0
```

If `face_model_dataset` is missing, the notebook falls back to `Project3/dataset`.

### Preprocessing Pipeline (matches `train.py` exactly)

| Step | Code in `train.py` |
|---|---|
| Grayscale | `cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)` |
| Resize to 96×96 | `cv2.resize(gray, (IMG_SIZE, IMG_SIZE))` |
| Normalize to [0,1] | `resized / 255.0` |

This exact pipeline is also applied in `testing.py` and mirrored in `main.cpp` (the BT.601 luma conversion is mathematically equivalent to OpenCV's grayscale conversion for RGB565 input).

In [3]:
# ============================================================
# CELL 2 — Configuration and dataset check
# Run this after the Imports cell every time you open the notebook.
# ============================================================

# ── Config — matches train.py exactly ────────────────────────
IMG_SIZE = 96
EPOCHS = 20
BATCH_SIZE = 32

# Resolve the Project3 folder whether notebook is run from repo root or Project3.
cwd = Path.cwd().resolve()
PROJECT3_DIR = (cwd / "Project3").resolve() if (cwd / "Project3").is_dir() else cwd

DATASET_CANDIDATES = [
    PROJECT3_DIR / "face_model_dataset",
    PROJECT3_DIR / "dataset",
    PROJECT3_DIR / "FaceDetection" / "dataset",
]

DATASET_DIR = None
for candidate in DATASET_CANDIDATES:
    if (candidate / "face").is_dir() and (candidate / "noface").is_dir():
        DATASET_DIR = candidate
        break

if DATASET_DIR is None:
    DATASET_DIR = DATASET_CANDIDATES[0]

FIGURES_DIR = PROJECT3_DIR / "figures"
TRAINING_DIR = PROJECT3_DIR / "training"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TRAINING_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project3 dir : {PROJECT3_DIR}")
print(f"Dataset dir  : {DATASET_DIR}")
print(f"Figures dir  : {FIGURES_DIR}")
print(f"Training dir : {TRAINING_DIR}")
print()

# ── Dataset check ─────────────────────────────────────────────
print("Dataset status:")
for cls in ["face", "noface"]:
    cls_dir = DATASET_DIR / cls
    files = (
        [f for f in os.listdir(cls_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))]
        if cls_dir.exists()
        else []
    )
    status = "✓ OK" if len(files) >= 50 else ("⚠ LOW" if files else "✗ EMPTY")
    print(f"  {cls_dir}  →  {len(files)} images  {status}")


Project3 dir : /home/joshnelson/Documents/gcu/compsci/CST-440-MISC/CST-440/Project3
Dataset dir  : /home/joshnelson/Documents/gcu/compsci/CST-440-MISC/CST-440/Project3/face_model_dataset
Figures dir  : /home/joshnelson/Documents/gcu/compsci/CST-440-MISC/CST-440/Project3/figures
Training dir : /home/joshnelson/Documents/gcu/compsci/CST-440-MISC/CST-440/Project3/training

Dataset status:
  /home/joshnelson/Documents/gcu/compsci/CST-440-MISC/CST-440/Project3/face_model_dataset/face  →  0 images  ✗ EMPTY
  /home/joshnelson/Documents/gcu/compsci/CST-440-MISC/CST-440/Project3/face_model_dataset/noface  →  0 images  ✗ EMPTY


### Cell 2A — Verify Existing Dataset Folders

This notebook does **not** collect photos from a webcam. It trains directly from files already stored in `face_model_dataset` (or fallback `dataset`) inside the Project3 folder.

In [5]:
# ============================================================
# CELL 2A — Verify dataset folders and show sample filenames
# ============================================================

required_dirs = [DATASET_DIR / "face", DATASET_DIR / "noface"]
missing = [str(p) for p in required_dirs if not p.exists()]

if missing:
    message = "Missing required dataset directories:"
    message += os.linesep + os.linesep.join(missing)
    message += os.linesep * 2 + "Expected Project3/face_model_dataset/{face,noface} (or fallback dataset/{face,noface})."
    raise FileNotFoundError(message)

print("Using existing dataset files. Webcam collection is disabled in this notebook.")
for cls in ["face", "noface"]:
    cls_dir = DATASET_DIR / cls
    files = sorted([f for f in os.listdir(cls_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))])
    print(f"{cls:>6}: {len(files)} files in {cls_dir}")
    if files:
        print(f"       sample: {files[:3]}")

FileNotFoundError: Missing required dataset directories:
/home/joshnelson/Documents/gcu/compsci/CST-440-MISC/CST-440/Project3/face_model_dataset/face
/home/joshnelson/Documents/gcu/compsci/CST-440-MISC/CST-440/Project3/face_model_dataset/noface

Expected Project3/face_model_dataset/{face,noface} (or fallback dataset/{face,noface}).

### Cell 2B — Dataset Snapshot (No New Collection)

This optional cell only previews a few existing images from each class to confirm the dataset is ready for training.

In [ ]:
# ============================================================
# CELL 2B — Preview a few existing images (no webcam capture)
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(8, 5))
fig.suptitle("Existing Dataset Preview (Project3)", fontweight="bold")

for row, cls in enumerate(["face", "noface"]):
    cls_dir = DATASET_DIR / cls
    files = sorted([f for f in os.listdir(cls_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))])[:3]
    for col in range(3):
        ax = axes[row, col]
        ax.axis("off")
        if col < len(files):
            img = cv2.imread(str(cls_dir / files[col]), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                ax.imshow(cv2.resize(img, (96, 96)), cmap="gray", vmin=0, vmax=255)
                ax.set_title(f"{cls}: {files[col]}", fontsize=8)

plt.tight_layout()
plt.show()

### Figure 1 — Class Distribution    |    Figure 2 — Sample Crops

In [ ]:
# ============================================================
# CELL 3 — Dataset statistics + Figures 1 and 2
# ============================================================

face_dir = DATASET_DIR / "face"
noface_dir = DATASET_DIR / "noface"

face_files = [f for f in os.listdir(face_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))] if face_dir.exists() else []
noface_files = [f for f in os.listdir(noface_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))] if noface_dir.exists() else []
counts = [len(face_files), len(noface_files)]

print(f"Dataset path: {DATASET_DIR}")
print(f"  face   : {counts[0]} images")
print(f"  noface : {counts[1]} images")
print(f"  TOTAL  : {sum(counts)} images")

if min(counts) == 0:
    print()
    print("⚠ One class is empty — add files to both class folders before continuing.")
else:
    # Figure 1 — class distribution bar chart
    fig, ax = plt.subplots(figsize=(5, 3))
    bars = ax.bar(["face", "noface"], counts, color=["steelblue", "#C44E52"], edgecolor="white", width=0.4)
    ax.set_ylabel("Image Count")
    ax.set_ylim(0, max(counts) * 1.3)
    ax.set_title("Figure 1 — Dataset Class Distribution", fontweight="bold")
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2, str(cnt), ha="center", fontweight="bold")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "fig1_dataset_distribution.png", dpi=150)
    plt.show()

    # Figure 2 — one sample per class
    fig2, axes = plt.subplots(1, 2, figsize=(5, 3))
    fig2.suptitle("Figure 2 — Sample 96×96 Grayscale Crop Per Class", fontweight="bold")
    for ax2, cls, files in zip(axes, ["face", "noface"], [face_files, noface_files]):
        ax2.set_title(cls)
        ax2.axis("off")
        if files:
            img = cv2.imread(str(DATASET_DIR / cls / files[0]), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                ax2.imshow(cv2.resize(img, (96, 96)), cmap="gray", vmin=0, vmax=255)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "fig2_sample_images.png", dpi=150)
    plt.show()

---

## 3. Model Design

### Architecture (identical to `train.py`)

Three Conv2D + MaxPool blocks extract progressively abstract features. No `padding="same"` is set, so Conv2D uses **valid padding** — spatial dimensions shrink slightly at each conv step before MaxPool. The Dense head has 32 neurons, and a single sigmoid output gives P(face).

```
Input (96×96×1)
  → Conv2D(8,  3×3, valid, ReLU) + MaxPool2D   →  47×47×8    edges
  → Conv2D(16, 3×3, valid, ReLU) + MaxPool2D   →  22×22×16   curves
  → Conv2D(32, 3×3, valid, ReLU) + MaxPool2D   →  10×10×32   facial structure
  → Flatten  →  Dense(32, ReLU)  →  Dense(1, Sigmoid)
```

**Output:** `P(face) ≥ 0.5` → **FACE** | `P(face) < 0.5` → **NO FACE**  
This threshold matches both `testing.py` (`THRESHOLD = 0.5`) and `main.cpp` (`prob >= 0.5f`).  
**Loss:** Binary cross-entropy — the correct loss for a two-class binary classifier.

In [ ]:
# ============================================================
# CELL 4 — Model definition (matches train.py exactly)
# ============================================================

model = models.Sequential([

    layers.Input(shape=(96, 96, 1)),

    layers.Conv2D(8,  3, activation="relu"),   # valid padding — matches train.py
    layers.MaxPool2D(),

    layers.Conv2D(16, 3, activation="relu"),
    layers.MaxPool2D(),

    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPool2D(),

    layers.Flatten(),
    layers.Dense(32, activation="relu"),       # 32 neurons — matches train.py
    layers.Dense(1,  activation="sigmoid"),    # single output: P(face)

], name="FaceDetector")

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

---

## 4. Training the Model

The cell below is a direct port of `train.py` into the notebook, with training curves added as Figure 3.

| Parameter | Value | Source |
|---|---|---|
| Optimizer | Adam | `train.py` |
| Loss | Binary Cross-Entropy | `train.py` |
| Epochs | 20 | `train.py` `EPOCHS = 20` |
| Batch Size | 32 | `train.py` `BATCH_SIZE = 32` |
| Dense neurons | 32 | `train.py` |
| Train / Test split | 80% / 20% | `train.py` |

In [ ]:
# ============================================================
# CELL 5 — Load dataset and train (mirrors train.py exactly)
# ============================================================

def load_images(folder, label):
    """Load all images from DATASET_DIR/<folder>."""
    data = []
    path = DATASET_DIR / folder
    if not path.exists():
        print(f"[WARN] {path} not found")
        return data

    for file in sorted(os.listdir(path)):
        if not file.lower().endswith((".png", ".jpg", ".jpeg")):
            continue
        img = cv2.imread(str(path / file))
        if img is None:
            continue
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)        # grayscale
        resized = cv2.resize(gray, (IMG_SIZE, IMG_SIZE))     # 96×96
        norm = resized / 255.0                               # normalize [0,1]
        data.append((norm, label))
    return data

print("[INFO] Loading dataset...")
face_data = load_images("face", 1)      # label 1 = face
noface_data = load_images("noface", 0)  # label 0 = no face
dataset = face_data + noface_data

if len(dataset) == 0:
    raise RuntimeError(f"No images loaded from {DATASET_DIR}. Check class folders.")
if len(face_data) == 0 or len(noface_data) == 0:
    raise RuntimeError(
        "Both classes required. "
        f"face={len(face_data)}  noface={len(noface_data)}. "
        f"Dataset directory in use: {DATASET_DIR}"
    )

np.random.shuffle(dataset)
X = np.array([d[0] for d in dataset], dtype=np.float32).reshape(-1, IMG_SIZE, IMG_SIZE, 1)
y = np.array([d[1] for d in dataset], dtype=np.float32)
print(f"Total: {len(X)} images  |  face: {int(y.sum())}  noface: {int((1-y).sum())}")

# Train / test split — no stratify, matches train.py
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(f"Train: {len(X_train)}   Test: {len(X_test)}")

# Build model fresh (same architecture as Cell 4 and train.py)
model = models.Sequential([
    layers.Input(shape=(96, 96, 1)),
    layers.Conv2D(8, 3, activation="relu"),
    layers.MaxPool2D(),
    layers.Conv2D(16, 3, activation="relu"),
    layers.MaxPool2D(),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPool2D(),
    layers.Flatten(),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
], name="FaceDetector")
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

print()
print("[INFO] Training...")
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print()
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc*100:.2f}%")

# ── Figure 3 — training curves ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Figure 3 — Training History", fontsize=13, fontweight="bold")

axes[0].plot(history.history["accuracy"], label="Train", color="steelblue", lw=2)
axes[0].plot(history.history["val_accuracy"], label="Validation", color="darkorange", lw=2, ls="--")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_ylim(0, 1.05)

axes[1].plot(history.history["loss"], label="Train", color="steelblue", lw=2)
axes[1].plot(history.history["val_loss"], label="Validation", color="darkorange", lw=2, ls="--")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig3_training_curves.png", dpi=150)
plt.show()
print("Figure 3 saved.")

---

## 5. Evaluation

In [ ]:
# ============================================================
# CELL 6 — Figures 4, 5, 6: confusion matrix,
#           precision/recall/F1, confidence distribution
# ============================================================

y_prob = model.predict(X_test, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)
y_true = y_test.astype(int)

print(f"Unique true labels : {np.unique(y_true)}")
print(f"Unique predictions : {np.unique(y_pred)}")
print(f"Predicted face     : {np.sum(y_pred == 1)}  |  noface: {np.sum(y_pred == 0)}")
print()
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=["no face", "face"], zero_division=0))

# Figure 4 — confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["no face", "face"])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=True, cmap="Blues")
ax.set_title("Figure 4 — Confusion Matrix (Test Set)", fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig4_confusion_matrix.png", dpi=150)
plt.show()

# Figure 5 — precision / recall / F1 per class
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0)
x, w = np.arange(2), 0.25
fig2, ax2 = plt.subplots(figsize=(7, 4))
ax2.bar(x - w, precision, w, label="Precision", color="steelblue", edgecolor="white")
ax2.bar(x, recall, w, label="Recall", color="darkorange", edgecolor="white")
ax2.bar(x + w, f1, w, label="F1 Score", color="#55A868", edgecolor="white")
ax2.axhline(0.80, color="black", ls="--", lw=1.2, label="80% threshold")
ax2.set_xticks(x)
ax2.set_xticklabels(["no face", "face"], fontsize=11)
ax2.set_ylabel("Score")
ax2.set_ylim(0, 1.15)
ax2.set_title("Figure 5 — Precision, Recall & F1 per Class", fontweight="bold")
ax2.legend(fontsize=9)
for i, (p, r, f) in enumerate(zip(precision, recall, f1)):
    ax2.text(i - w, p + 0.01, f"{p:.2f}", ha="center", fontsize=8)
    ax2.text(i, r + 0.01, f"{r:.2f}", ha="center", fontsize=8)
    ax2.text(i + w, f + 0.01, f"{f:.2f}", ha="center", fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig5_precision_recall_f1.png", dpi=150)
plt.show()

# Figure 6 — confidence distribution
face_p = y_prob[y_true == 1]
noface_p = y_prob[y_true == 0]
fig3, ax3 = plt.subplots(figsize=(8, 4))
if len(face_p) > 0:
    ax3.hist(face_p, bins=20, alpha=0.7, color="steelblue", label="Actual: face", edgecolor="white")
if len(noface_p) > 0:
    ax3.hist(noface_p, bins=20, alpha=0.7, color="#C44E52", label="Actual: no face", edgecolor="white")
ax3.axvline(0.5, color="black", ls="--", lw=1.5, label="Decision boundary (0.5)")
ax3.set_xlabel("P(face) — Model Output")
ax3.set_ylabel("Count")
ax3.set_title("Figure 6 — Prediction Confidence Distribution", fontweight="bold")
ax3.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig6_confidence_distribution.png", dpi=150)
plt.show()

print()
print(f"Test Accuracy : {test_acc*100:.2f}%")

### Evaluation Summary

| Metric | Value |
|---|---|
| Test Accuracy | *(fill in after running)* |
| Test Loss | *(fill in after running)* |

**Figure 4 — Confusion Matrix:**
- Top-left (TN): noface correctly rejected
- Bottom-right (TP): face correctly detected
- Top-right (FP): noface predicted as face
- Bottom-left (FN): face missed — predicted as noface

**Figure 5 — Precision / Recall / F1:** Breaks down model quality per class beyond overall accuracy. Recall for the face class is the most important metric for a detection system — it measures how many real faces were caught.

**Figure 6 — Confidence Distribution:** A well-trained binary model pushes face images toward P(face) ≈ 1.0 (right) and noface images toward P(face) ≈ 0.0 (left), with a clear gap at the 0.5 decision boundary. Overlap near 0.5 indicates uncertain predictions.

---

## 6. Model Conversion to TensorFlow Lite

Three output files are produced in `Project3/training`, matching `train.py` output names:

| File | Format | Used by |
|---|---|---|
| `face_model.tflite` | INT8 quantized | Arduino (`main.cpp` via model header) |
| `face_model_float32.tflite` | float32 | `testing.py` on PC |
| `face_model.h` | C byte array (`xxd -i`) | included by embedded firmware |

### INT8 Quantization

Weights are mapped from float32 to int8 using `scale` and `zero_point` calibrated by 100 representative training samples:

```
float_value = (int8_value − zero_point) × scale
```

`main.cpp` applies this dequantization at inference time:
```cpp
const float prob = (raw - zero_point) * scale;
return (prob >= 0.5f) ? 1 : 0;
```

The result is ~4× smaller model that runs 2–4× faster on the ARM Cortex-M4's integer ALU, with typically less than 2% accuracy loss.

In [ ]:
# ============================================================
# CELL 7 — INT8 quantization + float32 + C header
#           (matches train.py exactly)
# ============================================================

INT8_MODEL_PATH = TRAINING_DIR / "face_model.tflite"
FLOAT_MODEL_PATH = TRAINING_DIR / "face_model_float32.tflite"
HEADER_PATH = TRAINING_DIR / "face_model.h"


def rep_data():
    """Representative training samples for INT8 calibration."""
    for i in range(min(100, len(X_train))):
        yield [X_train[i : i + 1].astype(np.float32)]

# ── INT8 model (Arduino) ──────────────────────────────────────
print("[INFO] Converting to INT8 TFLite...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = rep_data
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

tflite_model = converter.convert()
with open(INT8_MODEL_PATH, "wb") as f:
    f.write(tflite_model)
print(f"[DONE] {INT8_MODEL_PATH}  →  {len(tflite_model) / 1024:.1f} KB")

# ── Float32 model (PC / testing.py) ──────────────────────────
float_converter = tf.lite.TFLiteConverter.from_keras_model(model)
float_tflite = float_converter.convert()
with open(FLOAT_MODEL_PATH, "wb") as f:
    f.write(float_tflite)
print(f"[DONE] {FLOAT_MODEL_PATH}  →  {len(float_tflite) / 1024:.1f} KB")

# ── C header for embedded firmware ────────────────────────────
with open(HEADER_PATH, "w", encoding="utf-8") as out:
    subprocess.run(["xxd", "-i", str(INT8_MODEL_PATH)], check=True, stdout=out)
print(f"[DONE] {HEADER_PATH} generated")

size_fp32 = len(float_tflite) / 1024
size_int8 = len(tflite_model) / 1024
print()
print(f"Size: {size_fp32:.1f} KB → {size_int8:.1f} KB  ({size_fp32 / size_int8:.1f}× reduction)")


In [ ]:
# ============================================================
# CELL 8 — Figure 7: Quantization impact (size + accuracy)
# ============================================================

def eval_tflite(path, X, y):
    """Run TFLite inference on test set, return accuracy."""
    interp = tf.lite.Interpreter(model_path=str(path))
    interp.allocate_tensors()
    ind = interp.get_input_details()[0]
    outd = interp.get_output_details()[0]
    sc, zp = outd["quantization"]
    correct = 0

    for img, label in zip(X, y):
        inp = (
            (img * 255).clip(0, 255).astype(np.uint8)
            if ind["dtype"] == np.uint8
            else img.astype(np.float32)
        ).reshape(1, 96, 96, 1)
        interp.set_tensor(ind["index"], inp)
        interp.invoke()
        raw = float(interp.get_tensor(outd["index"])[0][0])
        prob = max(0.0, min(1.0, (raw - zp) * sc if outd["dtype"] == np.uint8 else raw))
        if (1 if prob >= 0.5 else 0) == int(label):
            correct += 1

    return correct / len(y)

int8_acc = eval_tflite(INT8_MODEL_PATH, X_test, y_test)
print(f"Keras float32 : {test_acc*100:.2f}%")
print(f"TFLite INT8   : {int8_acc*100:.2f}%")
print(f"Accuracy drop : {(test_acc - int8_acc)*100:.2f}%")

COLORS = ["#DD8452", "#4C72B0"]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle("Figure 7 — Quantization Impact", fontsize=13, fontweight="bold")

b = axes[0].bar(
    ["float32\n(Keras / PC)", "INT8\n(Arduino)"],
    [size_fp32, size_int8],
    color=COLORS,
    edgecolor="white",
    width=0.4,
)
axes[0].set_ylabel("Model Size (KB)")
axes[0].set_title("Model Size")
axes[0].set_ylim(0, size_fp32 * 1.5)
for bar, sz in zip(b, [size_fp32, size_int8]):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3, f"{sz:.1f} KB", ha="center", fontweight="bold")

b2 = axes[1].bar(
    ["float32\n(Keras)", "INT8\n(TFLite)"],
    [test_acc * 100, int8_acc * 100],
    color=COLORS,
    edgecolor="white",
    width=0.4,
)
axes[1].axhline(80, color="black", ls="--", lw=1.2, label="80% requirement")
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_ylim(0, 110)
axes[1].set_title("Accuracy: float32 vs INT8")
axes[1].legend()
for bar, a in zip(b2, [test_acc * 100, int8_acc * 100]):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5, f"{a:.1f}%", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig7_quantization_impact.png", dpi=150)
plt.show()
print("Figure 7 saved.")

---

## 7. Testing the TensorFlow Lite Model

### Desktop Validation — `testing.py`

`testing.py` validates the INT8 model on a live webcam feed before flashing to the Arduino. It mirrors the exact inference pipeline in `main.cpp`:

1. Haar Cascade detects and crops the face region to 96×96 grayscale
2. Raw `uint8` pixels passed directly to the INT8 TFLite model
3. Output dequantized: `prob = (raw_uint8 - zero_point) × scale` (matches `main.cpp` line-for-line)
4. `prob ≥ 0.5` → **FACE** (green box) | `prob < 0.5` → **NO FACE** (red text)

Run from terminal in the project folder:
```bash
python training/testing.py
```
The first 10 frames print debug output: `[DEBUG] frame=N  prob=0.XXXX`  
Press **Q** to quit the webcam window.

When tested against a live webcam, `testing.py` displayed a **green bounding box** labeled **"FACE 99.6%"** around the detected face in a classroom setting, with a 96×96 grayscale crop inset shown in the top-left corner. When the camera was tilted away from any face, the window showed **red text "NO FACE DETECTED 0.0%"** with no bounding box drawn.

### On-Device Deployment — `main.cpp`

The `face_model.tflite` and generated `face_model.h` are included in the PlatformIO project. `main.cpp` runs this pipeline on every loop iteration:

```
OV2640 FIFO → stream 160×120 RGB565 pixel by pixel
  → center-crop 96×96 (CROP_X=32, CROP_Y=12 from model_config.h)
  → RGB565 → grayscale: Y = (77·R + 150·G + 29·B) >> 8  (BT.601)
  → write directly into inputTensor->data.uint8  [no frame buffer]
  → Invoke()
  → prob = (raw_uint8 - zero_point) × scale
  → prob ≥ 0.5f → Serial: "FACE DETECTED  prob=X.XXX  (N ms)"
  → prob < 0.5f → Serial: "No face detected  (N ms)"
```

Memory is kept within the 256 KB SRAM budget:
- No 160×120 frame buffer (saves ~38 KB)
- Tensor arena: 150 KB (`TENSOR_ARENA_KB=150` in `platformio.ini`)

**Actual Serial Monitor output (Arduino Nano 33 BLE Sense):**
```
15:35:29.603 > No face detected  (731 ms)
15:35:32.602 > No face detected  (731 ms)
15:35:35.601 > No face detected  (730 ms)
15:35:38.600 > >>> FACE DETECTED  prob=0.582  (731 ms)
15:35:41.600 > >>> FACE DETECTED  prob=0.582  (730 ms)
15:35:44.599 > >>> FACE DETECTED  prob=0.582  (731 ms)
15:35:47.599 > No face detected  (730 ms)
15:35:50.598 > No face detected  (731 ms)
15:35:53.597 > No face detected  (730 ms)
15:35:56.596 > >>> FACE DETECTED  prob=0.500  (731 ms)
15:35:59.596 > >>> FACE DETECTED  prob=0.500  (731 ms)
15:36:02.595 > No face detected  (730 ms)
15:36:05.593 > No face detected  (730 ms)
15:36:08.596 > No face detected  (730 ms)
```

Inference latency is consistently **~731 ms** per frame on the ARM Cortex-M4 @ 64 MHz. The model correctly alternates between `FACE DETECTED` and `No face detected` as a face moves in and out of frame.

### Iterative Improvement Log

| Version | Change | Result |
|---|---|---|
| v1.0 | Initial model, noface class empty (cascade bug) | Could not train |
| v1.1 | Fixed noface collection — bypass cascade, skip frames with faces | Both classes populated |
| v1.2 | 20 epochs, 150 images per class | ~87% validation accuracy |
| v1.3 | Added variety to noface (different backgrounds + lighting) | ~90%+ validation accuracy |


---

### Camera Accuracy Test — `camera_accuracy_test.py`

`training/camera_accuracy_test.py` benchmarks the deployed INT8 model against **your own camera** in real time. You act as the ground-truth labeler: press a key for each frame you want to score, and the script computes accuracy, precision, recall, F1, and a confusion matrix at the end. The code cell below contains the full script and can also be run directly from `Project3/training/`.

#### Setup

Make sure `face_model.tflite` exists in `Project3/training/` (run Cells 7 onwards first if it does not).

```bash
cd Project3/training
python camera_accuracy_test.py
```

Optional flags:

| Flag | Default | Purpose |
|---|---|---|
| `--model` | `face_model.tflite` | Path to a different `.tflite` file |
| `--camera` | `0` | OpenCV camera index (try `1` or `2` for external webcams) |
| `--threshold` | `0.5` | Decision boundary (matches `main.cpp`) |
| `--save` | `camera_accuracy_report.png` | Output filename for the PNG report |

#### How to Label Frames

The script shows a live window. A green/red box marks the **96×96 centre crop** that the model actually sees — the same crop that `main.cpp` cuts from the OV2640 frame.

| Key | Meaning |
|---|---|
| **F** | This frame is a real face → scored as ground truth = 1 |
| **N** | This frame has no face → scored as ground truth = 0 |
| **S** or spacebar | Skip (do not score this frame) |
| **Q** | Quit and show the accuracy report |

#### Recommended Test Protocol

1. **Face frames (≥ 20):** Sit in front of the camera, center your face in the green box, and press **F** for each frame. Vary your distance, angle, and lighting slightly between presses.
2. **No-face frames (≥ 20):** Move out of frame or cover the lens, then press **N**. Include varied backgrounds — empty walls, plants, objects — to test false-positive rejection.
3. **Aim for a balanced sample** (~50% face, ~50% no-face) so accuracy is not dominated by one class.
4. Press **Q** when done.

> Camera accuracy test results are located in Section 8 alongside the live demonstration and microcontroller output.


In [ ]:
# ============================================================
# CELL 9 — Live Camera Accuracy Test
# Runs camera_accuracy_test.py inline.
#
# Controls (click the OpenCV window first):
#   F  — label current frame as FACE    (ground truth = 1)
#   N  — label current frame as NO FACE (ground truth = 0)
#   S  — skip frame (not scored)
#   Q  — quit and print accuracy report + save PNG
#
# Prerequisites: face_model.tflite must exist in TRAINING_DIR
#   (run Cells 7-8 first).
# ============================================================

import time

# ── Config ────────────────────────────────────────────────────
CAM_INDEX  = 0      # change to 1/2 for an external webcam
THRESHOLD  = 0.5    # matches main.cpp threshold
SAVE_NAME  = "camera_accuracy_report.png"
MODEL_PATH = TRAINING_DIR / "face_model.tflite"

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model not found: {MODEL_PATH}\n"
        "Run Cells 7-8 first to generate face_model.tflite."
    )

# ── Load TFLite interpreter ───────────────────────────────────
print(f"[INFO] Loading model: {MODEL_PATH}")
_interp = tf.lite.Interpreter(model_path=str(MODEL_PATH))
_interp.allocate_tensors()

_in_det  = _interp.get_input_details()[0]
_out_det = _interp.get_output_details()[0]
_SZ      = int(_in_det["shape"][1])   # 96

print(f"  Input  shape : {_in_det['shape']}  dtype: {_in_det['dtype'].__name__}")
print(f"  Output dtype : {_out_det['dtype'].__name__}")
print(f"  Threshold    : {THRESHOLD}")

# ── Inference helper ──────────────────────────────────────────
def _infer(gray96: np.ndarray) -> float:
    if _in_det["dtype"] == np.uint8:
        inp = gray96.reshape(1, _SZ, _SZ, 1).astype(np.uint8)
    else:
        inp = (gray96 / 255.0).reshape(1, _SZ, _SZ, 1).astype(np.float32)
    _interp.set_tensor(_in_det["index"], inp)
    _interp.invoke()
    raw = _interp.get_tensor(_out_det["index"])
    sc, zp = _out_det["quantization"]
    if _out_det["dtype"] == np.uint8:
        return float(np.clip((int(raw[0][0]) - zp) * sc, 0.0, 1.0))
    return float(np.clip(raw[0][0], 0.0, 1.0))

# ── Accuracy tracker ──────────────────────────────────────────
_y_true: list = []
_y_pred: list = []
_y_prob: list = []

def _tally():
    n = len(_y_true)
    c = sum(t == p for t, p in zip(_y_true, _y_pred))
    nf = sum(t == 1 for t in _y_true)
    nn = sum(t == 0 for t in _y_true)
    print(f"  Labeled: {n}  face={nf}  noface={nn}  accuracy={c/n*100:.1f}%")

# ── HUD overlay ───────────────────────────────────────────────
_FONT  = cv2.FONT_HERSHEY_SIMPLEX
_GREEN = (0, 220, 0);  _RED = (0, 0, 220)
_GRAY  = (160, 160, 160);  _WHITE = (255, 255, 255);  _BLACK = (0, 0, 0)

def _hud(frame, prob, label, n_labeled, acc_pct):
    h, w = frame.shape[:2]
    color = _GREEN if label == "FACE" else _RED
    cv2.rectangle(frame, (0, 0), (w, 45), _BLACK, -1)
    cv2.putText(frame, f"{label}   p={prob:.3f}", (10, 32),
                _FONT, 1.0, color, 2, cv2.LINE_AA)
    cv2.rectangle(frame, (0, h - 50), (w, h), _BLACK, -1)
    acc_str = f"{acc_pct:.1f}%" if n_labeled > 0 else "---"
    cv2.putText(frame, f"Labeled: {n_labeled}   Live accuracy: {acc_str}",
                (10, h - 18), _FONT, 0.7, _WHITE, 1, cv2.LINE_AA)
    hints = "[F]=face  [N]=noface  [S]=skip  [Q]=quit"
    (tw, _), _ = cv2.getTextSize(hints, _FONT, 0.55, 1)
    cv2.putText(frame, hints, (w - tw - 10, 32), _FONT, 0.55, _GRAY, 1, cv2.LINE_AA)

# ── Camera loop ───────────────────────────────────────────────
print()
print("=" * 56)
print(" LIVE CAMERA ACCURACY TEST")
print("=" * 56)
print("  F  — label frame as FACE    (ground truth = 1)")
print("  N  — label frame as NO FACE (ground truth = 0)")
print("  S  — skip this frame")
print("  Q  — quit and show accuracy report")
print("=" * 56)

cap = cv2.VideoCapture(CAM_INDEX)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open camera index {CAM_INDEX}.")

for _ in range(5):   # warm-up frames
    cap.read()

WINDOW = "Camera Accuracy Test — F=face  N=noface  S=skip  Q=quit"
cv2.namedWindow(WINDOW, cv2.WINDOW_NORMAL)
cv2.resizeWindow(WINDOW, 900, 680)

flash_color = None
flash_until = 0.0

while True:
    ret, frame = cap.read()
    if not ret:
        print("[ERROR] Failed to read camera frame.")
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    fh, fw = gray.shape
    cy, cx = fh // 2, fw // 2
    half   = _SZ // 2
    crop   = cv2.resize(gray[max(0, cy-half):cy+half,
                             max(0, cx-half):cx+half], (_SZ, _SZ))

    prob  = _infer(crop)
    label = "FACE" if prob >= THRESHOLD else "NO FACE"
    bcolor = _GREEN if prob >= THRESHOLD else _RED
    cv2.rectangle(frame, (cx-half, cy-half), (cx+half, cy+half), bcolor, 2)

    now = time.time()
    if flash_color and now < flash_until:
        overlay = frame.copy()
        overlay[:, :] = flash_color
        cv2.addWeighted(overlay, 0.25, frame, 0.75, 0, frame)

    n_lab = len(_y_true)
    corr  = sum(t == p for t, p in zip(_y_true, _y_pred))
    _hud(frame, prob, label, n_lab, corr / n_lab * 100 if n_lab else 0.0)
    cv2.imshow(WINDOW, frame)

    key = cv2.waitKey(1) & 0xFF
    if key in (ord("q"), ord("Q")):
        break
    elif key in (ord("f"), ord("F")):
        pred = 1 if prob >= THRESHOLD else 0
        _y_true.append(1); _y_pred.append(pred); _y_prob.append(prob)
        flash_color = (0, 180, 0); flash_until = time.time() + 0.25
        ok = "✓" if pred == 1 else "✗"
        print(f"  [F] frame {n_lab+1:3d}  prob={prob:.3f}  pred={label}  {ok}")
        if (n_lab + 1) % 10 == 0:
            _tally()
    elif key in (ord("n"), ord("N")):
        pred = 1 if prob >= THRESHOLD else 0
        _y_true.append(0); _y_pred.append(pred); _y_prob.append(prob)
        flash_color = (0, 0, 180); flash_until = time.time() + 0.25
        ok = "✓" if pred == 0 else "✗"
        print(f"  [N] frame {n_lab+1:3d}  prob={prob:.3f}  pred={label}  {ok}")
        if (n_lab + 1) % 10 == 0:
            _tally()

cap.release()
cv2.destroyAllWindows()

# ── Accuracy report ───────────────────────────────────────────
print()
print("=" * 56)
print(" ACCURACY REPORT")
print("=" * 56)

if len(_y_true) == 0:
    print("[WARN] No frames were labeled — nothing to report.")
else:
    _yt = np.array(_y_true)
    _yp = np.array(_y_pred)
    _yb = np.array(_y_prob)

    _total   = len(_yt)
    _correct = int((_yt == _yp).sum())
    _acc     = _correct / _total
    _n_face  = int((_yt == 1).sum())
    _n_nof   = int((_yt == 0).sum())

    print(f"  Total labeled frames : {_total}")
    print(f"    Face    (label=1)  : {_n_face}")
    print(f"    No Face (label=0)  : {_n_nof}")
    print(f"  Correct predictions  : {_correct}")
    print(f"  Overall accuracy     : {_acc*100:.2f}%")
    print()
    print(classification_report(_yt, _yp, target_names=["no face", "face"], zero_division=0))

    _cm = confusion_matrix(_yt, _yp, labels=[0, 1])
    print("Confusion matrix (rows=actual, cols=predicted):")
    print(f"            pred:noface  pred:face")
    print(f"  act:noface   {_cm[0,0]:5d}       {_cm[0,1]:5d}")
    print(f"  act:face     {_cm[1,0]:5d}       {_cm[1,1]:5d}")
    print("=" * 56)

    # ── Save PNG ──────────────────────────────────────────────
    _save = FIGURES_DIR / SAVE_NAME
    fig_r, axes_r = plt.subplots(1, 3, figsize=(15, 4))
    fig_r.suptitle(
        f"Camera Accuracy Report — {_total} frames  |  Accuracy: {_acc*100:.1f}%",
        fontsize=13, fontweight="bold",
    )
    ConfusionMatrixDisplay(_cm, display_labels=["no face", "face"]).plot(
        ax=axes_r[0], colorbar=False, cmap="Blues"
    )
    axes_r[0].set_title("Confusion Matrix")

    _pr, _re, _f1, _ = precision_recall_fscore_support(_yt, _yp, labels=[0, 1], zero_division=0)
    xr, wr = np.arange(2), 0.22
    axes_r[1].bar(xr - wr, _pr, wr, label="Precision",  color="steelblue",  edgecolor="white")
    axes_r[1].bar(xr,      _re, wr, label="Recall",     color="darkorange", edgecolor="white")
    axes_r[1].bar(xr + wr, _f1, wr, label="F1",         color="#55A868",    edgecolor="white")
    axes_r[1].axhline(0.80, color="black", ls="--", lw=1.2, label="80% target")
    axes_r[1].set_xticks(xr); axes_r[1].set_xticklabels(["no face", "face"])
    axes_r[1].set_ylim(0, 1.15); axes_r[1].set_title("Precision / Recall / F1")
    axes_r[1].legend(fontsize=8)
    for i, (p, r, f) in enumerate(zip(_pr, _re, _f1)):
        axes_r[1].text(i - wr, p + 0.02, f"{p:.2f}", ha="center", fontsize=7)
        axes_r[1].text(i,      r + 0.02, f"{r:.2f}", ha="center", fontsize=7)
        axes_r[1].text(i + wr, f + 0.02, f"{f:.2f}", ha="center", fontsize=7)

    fp_ = _yb[_yt == 1]; nfp_ = _yb[_yt == 0]
    if len(fp_)  > 0: axes_r[2].hist(fp_,  bins=15, alpha=0.7, color="steelblue", label="Actual: face",    edgecolor="white")
    if len(nfp_) > 0: axes_r[2].hist(nfp_, bins=15, alpha=0.7, color="#C44E52",   label="Actual: no face", edgecolor="white")
    axes_r[2].axvline(THRESHOLD, color="black", ls="--", lw=1.5, label=f"Threshold ({THRESHOLD})")
    axes_r[2].set_xlabel("P(face)"); axes_r[2].set_ylabel("Frame count")
    axes_r[2].set_title("Confidence Distribution"); axes_r[2].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(_save, dpi=150)
    plt.show()
    print(f"\n[SAVED] Report PNG → {_save}")


---

## 8. Conclusion

### Results Summary

| Metric | Value |
|---|---|
| Test Accuracy (Keras float32) | *(fill in after running Cell 6)* |
| Test Loss (Keras float32) | *(fill in after running Cell 6)* |
| TFLite INT8 Accuracy | *(fill in after running Cell 8)* |
| Accuracy Drop (float32 → INT8) | *(fill in after running Cell 8)* |
| float32 Model Size | *(fill in after running Cell 7)* |
| INT8 Model Size | *(fill in after running Cell 7)* |
| On-Device Inference Latency | ~731 ms / frame |
| Camera Accuracy (pre-cleanup) | 87.5% (14/16 frames, imbalanced 15:1 sample) |
| Camera Accuracy (post-cleanup) | **100%** (16/16 frames, balanced 9 face / 7 noface) |

---

### Live Demonstration

> 🎥 **https://www.youtube.com/shorts/OUN4EUnCQx4**
>
> The video below shows the Arduino running live face detection via the Serial Monitor. Notice that while the model correctly fires `FACE DETECTED` when a face is present and `No face detected` when there isn't one, it is **not perfectly reliable in practice** — you will see frames where detection lags, drops out briefly, or triggers near the decision boundary (`prob ≈ 0.500`). This is expected behavior for a small embedded model.

---

### Camera Accuracy Test Results

These results were produced by `camera_accuracy_test.py` running against the deployed INT8 `face_model.tflite` — the same model file flashed to the Arduino.

#### Preliminary Results (before bad-data removal)

16 labeled frames, heavily imbalanced sample (15 face, 1 no-face), collected before cleaning the training dataset. The model achieved **87.5% overall accuracy** and perfect precision on the face class.

```
========================================================
 ACCURACY REPORT
========================================================
  Total labeled frames : 16
    Face    (label=1)  : 15
    No Face (label=0)  : 1
  Correct predictions  : 14
  Overall accuracy     : 87.50%

              precision    recall  f1-score   support

     no face       0.33      1.00      0.50         1
        face       1.00      0.87      0.93        15

    accuracy                           0.88        16
   macro avg       0.67      0.93      0.71        16
weighted avg       0.96      0.88      0.90        16

Confusion matrix (rows=actual, cols=predicted):
            pred:noface  pred:face
  act:noface       1           0
  act:face         2          13
========================================================
```

The 2 false negatives were traced to frames where the subject's face was partially outside the 96×96 crop box.

#### Final Results (after bad-data removal)

After removing low-quality training images and retraining, the model was re-evaluated with a balanced 16-frame sample (9 face, 7 no-face). The model achieved **100% accuracy** with perfect precision, recall, and F1 across both classes.

```
========================================================
 ACCURACY REPORT
========================================================
  Total labeled frames : 16
    Face    (label=1)  : 9
    No Face (label=0)  : 7
  Correct predictions  : 16
  Overall accuracy     : 100.00%

              precision    recall  f1-score   support

     no face       1.00      1.00      1.00         7
        face       1.00      1.00      1.00         9

    accuracy                           1.00        16
   macro avg       1.00      1.00      1.00        16
weighted avg       1.00      1.00      1.00        16

Confusion matrix (rows=actual, cols=predicted):
            pred:noface  pred:face
  act:noface       7           0
  act:face         0           9
========================================================
```

> **Note on the 100% result:** This score reflects a small, controlled 16-frame sample collected under favorable conditions. It does **not** mean the model is truly 100% accurate in general use. In real-world conditions — varied lighting, partial occlusion, faces at extreme angles — the model will make mistakes. The 100% figure confirms the dataset cleanup worked, not that the model is perfect.

> *Insert a screenshot of the terminal report and the saved PNG here.*

---

### Why isn't it 100% accurate in reality?

Several factors cause real-world accuracy to fall short of the controlled test result:

- **Small, constrained dataset.** The model was trained on a limited number of 96×96 grayscale images collected under similar conditions. It has not seen the full diversity of real faces (angles, skin tones, glasses, lighting variations).
- **Low resolution + grayscale.** Downsampling to 96×96 and stripping color discards a lot of discriminating information.
- **Fixed 96×96 center crop.** `main.cpp` always cuts the same center region of the OV2640's 160×120 frame. If a face drifts to the edge, it falls outside the crop and the model never sees it.
- **INT8 quantization.** Mapping float32 weights to 8-bit integers introduces small rounding errors that occasionally flip a borderline prediction.
- **~731 ms inference latency.** At roughly 1.4 frames per second, fast motion can carry a face through the crop window between readings.

---

### On-Device Firmware — `FaceDetection/`

The `FaceDetection/` PlatformIO project contains all the embedded code that runs on the Arduino Nano 33 BLE Sense. The three source files below are included in full for reference.

#### `platformio.ini` — Board Configuration & Dependencies

```ini
; PlatformIO Configuration for Face Detection (96x96 grayscale)
; Arduino Nano 33 BLE Sense (nRF52840, 256 KB SRAM)
; Camera: ArduCAM Mini 2MP Plus (OV2640) — SPI + I2C interface
;
; Wiring (OV2640 → Nano 33 BLE Sense):
;   VCC   → 5V   (onboard AMS1117-3.3 LDO steps down to 3.3V for the sensor)
;   GND   → GND
;   SDA   → A4   (3.3V I2C)
;   SCL   → A5   (3.3V I2C)
;   MOSI  → 11   (3.3V SPI)
;   MISO  → 12   (3.3V SPI)
;   SCK   → 13   (3.3V SPI)
;   CS    → 10   (configurable — see CAM_CS in main.cpp)

[env:nano33ble]
platform  = nordicnrf52
board     = nano33ble
framework = arduino

lib_deps =
    ArduCAM/ArduCAM @ ^1.0.0
    https://github.com/tensorflow/tflite-micro-arduino-examples#main

build_flags =
    -DOV2640_MINI_2MP_PLUS
    -DTF_LITE_STATIC_MEMORY
    -DTENSOR_ARENA_KB=150
    -O2

monitor_speed   = 115200
monitor_port    = /dev/ttyACM0
monitor_filters = time
extra_scripts   = extra_scripts.py
```

#### `src/model_config.h` — Image Dimensions & Class Names

```cpp
// Configuration for 96x96 face detection (TFLite Micro)
// Board:  Arduino Nano 33 BLE Sense (nRF52840, 256 KB SRAM)
// Camera: ArduCAM Mini 2MP Plus (OV2640) — SPI/I2C interface
// Must match training: train.py uses 96x96 grayscale, normalized [0,1], then quantized to uint8

#ifndef MODEL_CONFIG_H
#define MODEL_CONFIG_H

// Image dimensions - MUST MATCH TRAINING (train.py IMG_SIZE = 96)
#define IMAGE_WIDTH    96
#define IMAGE_HEIGHT   96
#define IMAGE_CHANNELS 1
#define IMAGE_SIZE     (IMAGE_WIDTH * IMAGE_HEIGHT * IMAGE_CHANNELS)

// Model: binary classification (face vs no-face)
#define NUM_CLASSES    2

// Class names (index 0 = no face, 1 = face; output is single sigmoid quantized to uint8)
const char* const CLASS_NAMES[] = {
  "no_face",
  "face",
};

#endif // MODEL_CONFIG_H
```

#### `src/main.cpp` — Full Firmware

```cpp
/*
 * Face Detection — Arduino Nano 33 BLE Sense + ArduCAM Mini 2MP Plus (OV2640)
 *
 * Camera wiring (OV2640 → Nano 33 BLE Sense):
 *   VCC   → 3.3V    (module has onboard AMS1117-3.3 LDO; 5V in → 3.3V to sensor)
 *   GND   → GND
 *   SDA   → A4    (3.3V I2C)
 *   SCL   → A5    (3.3V I2C)
 *   MOSI  → 11    (3.3V SPI)
 *   MISO  → 12    (3.3V SPI)
 *   SCK   → 13    (3.3V SPI)
 *   CS    → 10    (configurable — see CAM_CS below)
 *
 * Image pipeline:
 *   OV2640 QQVGA 160×120 RGB565 → stream FIFO row-by-row
 *   → convert RGB565 → grayscale (BT.601 integer luma)
 *   → center-crop to 96×96 written directly into TFLite input tensor
 *   No intermediate frame buffer needed — saves ~38 KB of SRAM.
 *
 * Model input:  uint8 [1,96,96,1]  (grayscale, quantized uint8)
 * Model output: uint8 (quantized sigmoid → dequantised to probability)
 */

#include <Arduino.h>
#include <Wire.h>
#include <SPI.h>
#include <memorysaver.h>
#include <ArduCAM.h>
#ifdef swap
#undef swap
#endif
#include <TensorFlowLite.h>
#include <tensorflow/lite/micro/all_ops_resolver.h>
#include <tensorflow/lite/micro/micro_interpreter.h>
#include <tensorflow/lite/schema/schema_generated.h>
#include "model_config.h"
#include "face_model_data.h"

#define CAM_CS     10
#define CAM_WIDTH  160
#define CAM_HEIGHT 120
#define CROP_X     ((CAM_WIDTH  - IMAGE_WIDTH)  / 2)   // 32 px
#define CROP_Y     ((CAM_HEIGHT - IMAGE_HEIGHT) / 2)   // 12 px

static ArduCAM myCAM(OV2640, CAM_CS);

static tflite::AllOpsResolver     resolver;
static const tflite::Model*       tflModel    = nullptr;
static tflite::MicroInterpreter*  interpreter = nullptr;
static TfLiteTensor*              inputTensor  = nullptr;
static TfLiteTensor*              outputTensor = nullptr;

#ifndef TENSOR_ARENA_KB
#  define TENSOR_ARENA_KB 6
#endif
constexpr int kTensorArenaSize = TENSOR_ARENA_KB * 1024;
static uint8_t tensorArena[kTensorArenaSize] __attribute__((aligned(16)));

bool initTFLite() {
  tflModel = tflite::GetModel(face_model_data);
  if (tflModel->version() != TFLITE_SCHEMA_VERSION) return false;
  static tflite::MicroInterpreter staticInterpreter(
      tflModel, resolver, tensorArena, kTensorArenaSize);
  interpreter = &staticInterpreter;
  if (interpreter->AllocateTensors() != kTfLiteOk) return false;
  inputTensor  = interpreter->input(0);
  outputTensor = interpreter->output(0);
  return true;
}

bool initCamera() {
  Wire.begin(); SPI.begin();
  pinMode(CAM_CS, OUTPUT); digitalWrite(CAM_CS, HIGH);
  uint8_t r1 = 0, r2 = 0;
  for (int i = 0; i < 3; i++) {
    myCAM.write_reg(ARDUCHIP_TEST1, 0x55); r1 = myCAM.read_reg(ARDUCHIP_TEST1);
    myCAM.write_reg(ARDUCHIP_TEST1, 0xAA); r2 = myCAM.read_reg(ARDUCHIP_TEST1);
    if (r1 == 0x55 && r2 == 0xAA) break;
    delay(200);
  }
  if (r1 != 0x55 || r2 != 0xAA) return false;
  myCAM.set_format(BMP);
  myCAM.InitCAM();
  myCAM.OV2640_set_JPEG_size(OV2640_160x120);
  delay(1000);
  return true;
}

bool captureFrame() {
  myCAM.flush_fifo(); myCAM.clear_fifo_flag(); myCAM.start_capture();
  const unsigned long deadline = millis() + 2000UL;
  while (!myCAM.get_bit(ARDUCHIP_TRIG, CAP_DONE_MASK)) {
    if (millis() > deadline) return false;
  }
  uint8_t* dst = inputTensor->data.uint8;
  myCAM.CS_LOW(); myCAM.set_fifo_burst();
  for (int row = 0; row < CAM_HEIGHT; row++) {
    const bool rowInCrop = (row >= CROP_Y) && (row < CROP_Y + IMAGE_HEIGHT);
    for (int col = 0; col < CAM_WIDTH; col++) {
      const uint8_t hi = SPI.transfer(0x00);
      const uint8_t lo = SPI.transfer(0x00);
      if (rowInCrop && (col >= CROP_X) && (col < CROP_X + IMAGE_WIDTH)) {
        const uint16_t px = ((uint16_t)hi << 8) | lo;
        const uint8_t  r  = (px >> 11) << 3;
        const uint8_t  g  = ((px >> 5) & 0x3F) << 2;
        const uint8_t  b  = (px & 0x1F) << 3;
        *dst++ = (uint8_t)(((uint16_t)77 * r + 150u * g + 29u * b) >> 8);
      }
    }
  }
  myCAM.CS_HIGH();
  return true;
}

int runInference() {
  if (interpreter->Invoke() != kTfLiteOk) return -1;
  const float   scale      = outputTensor->params.scale;
  const int32_t zero_point = outputTensor->params.zero_point;
  const float   prob       = (outputTensor->data.uint8[0] - zero_point) * scale;
  return (prob >= 0.5f) ? 1 : 0;
}

void setup() {
  pinMode(LED_BUILTIN, OUTPUT);
  for (int i = 0; i < 6; i++) { digitalWrite(LED_BUILTIN, i % 2); delay(150); }
  Serial.begin(115200); delay(2000);
  Serial.println(F("\n================================"));
  Serial.println(F(" Face Detection — OV2640 + Nano33BLE"));
  Serial.println(F("================================"));
  if (!initCamera()) while (1) { digitalWrite(LED_BUILTIN, HIGH); delay(100); digitalWrite(LED_BUILTIN, LOW); delay(100); }
  if (!initTFLite()) while (1) { digitalWrite(LED_BUILTIN, HIGH); delay(300); digitalWrite(LED_BUILTIN, LOW); delay(300); }
  Serial.println(F("Ready — running inference every second."));
}

void loop() {
  if (!captureFrame()) { delay(1000); return; }
  const unsigned long t0     = millis();
  const int           result = runInference();
  const unsigned long dt     = millis() - t0;
  if (result == 1) {
    Serial.print(F(">>> FACE DETECTED  prob="));
    Serial.print((outputTensor->data.uint8[0] - outputTensor->params.zero_point)
                  * outputTensor->params.scale, 3);
    Serial.print(F("  (")); Serial.print(dt); Serial.println(F(" ms)"));
  } else if (result == 0) {
    Serial.print(F("No face detected  ("));
    Serial.print(dt); Serial.println(F(" ms)"));
  }
  delay(1000);
}
```

#### Deployment Steps

1. Copy `face_model.tflite` from `Project3/training/` and regenerate the C header:
   ```bash
   cd Project3/training
   xxd -i face_model.tflite \
     | sed 's/unsigned char face_model_tflite/alignas(8) const unsigned char face_model_data/g; \
            s/unsigned int face_model_tflite_len/const unsigned int face_model_data_len/g' \
     > ../FaceDetection/src/face_model_data.h
   ```
2. Open `FaceDetection/` in VS Code with the PlatformIO extension installed.
3. Click **Upload** (→ button in the PlatformIO toolbar), or run `pio run --target upload`.
4. Open the Serial Monitor at **115200 baud** (`pio device monitor --baud 115200`).

#### Troubleshooting

| Symptom | Likely Cause | Fix |
|---|---|---|
| `ArduCAM SPI not responding` | Bad SPI wiring | Check MOSI→11, MISO→12, SCK→13, CS→10 |
| `OV2640 not found` | Bad I2C wiring | Check SDA→A4, SCL→A5 |
| `AllocateTensors failed` | Arena too small | Increase `TENSOR_ARENA_KB` in `platformio.ini` |
| `Model schema version mismatch` | Stale model header | Re-run the `xxd` command above |
| Always prints `No face detected` | Wrong crop / quantization | Retrain and regenerate header |
| Upload fails | Port not found | Press reset button as upload begins |

---

### Summary

This project demonstrates a complete TinyML pipeline — from an existing on-disk dataset through real-time on-device binary face detection — on an Arduino Nano 33 BLE Sense with an ArduCAM OV2640. The system classifies each camera frame as FACE or NO FACE, satisfying the ≥80% accuracy requirement.

### Key Findings

**Using a fixed dataset directory improves reproducibility.** The notebook reads `Project3/face_model_dataset` (with a fallback to `Project3/dataset`) and trains directly from those files, avoiding webcam collection steps during notebook execution.

**Preprocessing consistency is the most critical requirement.** `train.py`, `testing.py`, and `main.cpp` all apply the same pipeline: grayscale → 96×96 → normalize. The `model_config.h` header makes these dimensions an explicit contract across all three components.

**INT8 quantization is effective with minimal accuracy loss.** The ~4× size reduction from float32 to INT8 is essential for fitting within the Nano's 1 MB flash, with typically less than 2 percentage points of accuracy loss.

**Memory-efficient streaming eliminates the frame buffer.** `main.cpp` reads the OV2640 FIFO pixel-by-pixel and writes only the 96×96 center crop directly into the TFLite input tensor, saving ~38 KB of SRAM.

**Dataset quality matters more than quantity.** The camera accuracy test jumped from 87.5% to 100% after removing low-quality training images — without changing the model architecture or hyperparameters at all.

### Limitations and Future Work

- Expanding noface variety (rooms, lighting, object types) should further reduce false positives
- Data augmentation (brightness jitter, random flips) would reduce the training/validation accuracy gap without requiring more images
- Natural extension: multi-class person identification — expanding from face/no-face to recognizing which specific person is present

---

## References

Howard et al. (2017). *MobileNets: Efficient CNNs for Mobile Vision Applications.* arXiv:1704.04861

Jacob et al. (2018). Quantization and training of neural networks for efficient integer-arithmetic-only inference. *IEEE CVPR.*

TensorFlow. (2024). *TensorFlow Lite for Microcontrollers.* https://www.tensorflow.org/lite/microcontrollers

Viola & Jones. (2001). Rapid object detection using a boosted cascade of simple features. *IEEE CVPR.*

Warden & Situnayake. (2019). *TinyML: Machine Learning with TensorFlow Lite on Arduino.* O'Reilly Media.
